<a href="https://colab.research.google.com/github/nehalahane22-sketch/Voyage-Analytics-Integrating-MLOps-in-Travel/blob/main/Voyage_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Upload and Load Data
from google.colab import files
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

# Upload files (Select users.csv, flights.csv, hotels.csv)
uploaded = files.upload()

users_df = pd.read_csv('users.csv')
flights_df = pd.read_csv('flights.csv')
hotels_df = pd.read_csv('hotels.csv')

print(f"Users shape: {users_df.shape}")
print(f"Flights shape: {flights_df.shape}")
print(f"Hotels shape: {hotels_df.shape}")


Saving flights.csv to flights.csv
Saving hotels.csv to hotels.csv
Saving users.csv to users.csv
Users shape: (1340, 5)
Flights shape: (271888, 10)
Hotels shape: (40552, 8)


In [2]:
# Cell 2: Exploratory Data Analysis & Feature Preprocessing
# Parse dates
flights_df['date'] = pd.to_datetime(flights_df['date'])
flights_df['flight_month'] = flights_df['date'].dt.month
flights_df['flight_dayofweek'] = flights_df['date'].dt.dayofweek

hotels_df['date'] = pd.to_datetime(hotels_df['date'])
hotels_df['booking_month'] = hotels_df['date'].dt.month
hotels_df['booking_dayofweek'] = hotels_df['date'].dt.dayofweek

# Merge user demographic details with flights for richer regression signals
merged_flights = flights_df.merge(
    users_df[['code', 'gender', 'age', 'company']],
    left_on='userCode',
    right_on='code',
    how='left'
).drop(columns=['code'])

print(merged_flights.head(3))

   travelCode  userCode                from                  to  flightType  \
0           0         0         Recife (PE)  Florianopolis (SC)  firstClass   
1           0         0  Florianopolis (SC)         Recife (PE)  firstClass   
2           1         0       Brasilia (DF)  Florianopolis (SC)  firstClass   

     price  time  distance       agency       date  flight_month  \
0  1434.38  1.76    676.53  FlyingDrops 2019-09-26             9   
1  1292.29  1.76    676.53  FlyingDrops 2019-09-30             9   
2  1487.52  1.66    637.56      CloudFy 2019-10-03            10   

   flight_dayofweek gender  age company  
0                 3   male   21    4You  
1                 0   male   21    4You  
2                 3   male   21    4You  


In [11]:
# Cell 3: Install MLflow & Train Regression Models
!pip install mlflow pyngrok -q

import mlflow   # Install MLflow for experiment tracking and pyngrok for optional remote access
import mlflow.sklearn     # Import MLflow tools for tracking and saving Scikit-learn models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import os
import numpy as np

# Feature selection # Select travel and user features that may influence flight price
reg_features = ['from', 'to', 'flightType', 'time', 'distance', 'agency', 'flight_month', 'flight_dayofweek', 'age']
target = 'price'

# Separate input features (X) from the target price (y)
X = merged_flights[reg_features]
y = merged_flights[target]

# Split data into 80% training and 20% testing sets for unbiased evaluation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identify categorical features that require one-hot encoding
categorical_cols = ['from', 'to', 'flightType', 'agency']
# Identify numerical features that require scaling
numerical_cols = ['time', 'distance', 'flight_month', 'flight_dayofweek', 'age']

# Scale numerical features and safely one-hot encode categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

# Start MLflow Experiment # Create an MLflow experiment to track and compare flight price models
mlflow.set_experiment("Flight_Price_Prediction")


# Define Gradient Boosting and Random Forest models for performance comparison
models = {
    "GradientBoosting": GradientBoostingRegressor(n_estimators=120, max_depth=6, random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
}

# Initialize variables to keep track of the best-performing model
best_r2 = -1
best_pipeline = None

# Train and evaluate each candidate regression model
for name, model in models.items():
    with mlflow.start_run(run_name=name):
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('regressor', model)
        ])

        # Build a pipeline that preprocesses data and then trains the regression model
        pipeline.fit(X_train, y_train)
        preds = pipeline.predict(X_test)

        rmse = np.sqrt(mean_squared_error(y_test, preds))   # Calculate RMSE to measure the magnitude of prediction errors
        mae = mean_absolute_error(y_test, preds)    # Calculate average absolute prediction error using MAE
        r2 = r2_score(y_test, preds)      # Calculate R² to measure how well the model explains flight price variation

        mlflow.log_param("model_type", name)    # Record the model type in MLflow for experiment comparison
        mlflow.log_metric("rmse", rmse)   # Log the RMSE evaluation metric in MLflow
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2", r2)

        # Fixed: Explicitly use cloudpickle serialization format
        mlflow.sklearn.log_model(
            sk_model=pipeline,
            name=f"{name}_model",
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE
        )

        print(f"{name} -> RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.4f}")

        if r2 > best_r2:
            best_r2 = r2
            best_pipeline = pipeline

# Save winning pipeline locally
os.makedirs("models", exist_ok=True)
joblib.dump(best_pipeline, "models/flight_price_model.pkl")   # Save the best-performing flight price prediction pipeline for deployment

# Train and compare regression models, track experiments with MLflow, and save the best flight price prediction pipeline
print("\nBest Regression Pipeline saved to models/flight_price_model.pkl")



2026/08/30 05:29:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


GradientBoosting -> RMSE: 9.94, MAE: 7.51, R2: 0.9993


2026/08/30 05:30:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RandomForest -> RMSE: 25.85, MAE: 13.83, R2: 0.9949

Best Regression Pipeline saved to models/flight_price_model.pkl


In [12]:
# Cell 4: Gender Classification Model

# Import Random Forest for classification and metrics for evaluating class predictions
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Filter out 'none' labels # Keep only users with valid male/female labels for binary classification
gender_df = users_df[users_df['gender'].isin(['male', 'female'])].copy()

# Feature extraction: Extract user aggregate travel metrics
travel_stats = flights_df.groupby('userCode').agg(    # Group flight records by user to create user-level travel behavior features
    total_flights=('travelCode', 'count'),      # Count each user's total number of recorded flights
    avg_flight_price=('price', 'mean'),
    avg_distance=('distance', 'mean'),
    total_time=('time', 'sum')
).reset_index()

hotel_stats = hotels_df.groupby('userCode').agg(
    total_stays=('travelCode', 'count'),
    avg_hotel_price=('price', 'mean'),
    total_spent_hotels=('total', 'sum')
).reset_index()

# We need the gender label from users and travel features from flights in the same DataFrame.
user_features = gender_df.merge(travel_stats, left_on='code', right_on='userCode', how='left')
user_features = user_features.merge(hotel_stats, on='userCode', how='left').fillna(0)

# Select user, flight-behavior, and hotel-behavior features for classification
clf_features = ['age', 'company', 'total_flights', 'avg_flight_price', 'avg_distance', 'total_stays', 'avg_hotel_price']

# Separate classification inputs (X) from the gender target label (y)
X_clf = user_features[clf_features]
y_clf = user_features['gender']

# Create the classification preprocessor
# Scale numerical features and one-hot encode company before classification
clf_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['age', 'total_flights', 'avg_flight_price', 'avg_distance', 'total_stays', 'avg_hotel_price']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['company'])
    ]
)

# Create the complete ML pipeline
# Build one reusable pipeline containing preprocessing and the Random Forest classifier
gender_pipeline = Pipeline(steps=[
    ('preprocessor', clf_preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Split user data into training and testing sets for unbiased classifier evaluation
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)
gender_pipeline.fit(X_train_c, y_train_c)

preds_c = gender_pipeline.predict(X_test_c)
# Evaluate classifier performance using precision, recall, F1-score, and accuracy
print("Classification Report:\n", classification_report(y_test_c, preds_c))

# Save the trained gender classification pipeline for later inference or deployment
joblib.dump(gender_pipeline, "models/gender_classifier_model.pkl")
print("Gender Classifier saved to models/gender_classifier_model.pkl")

Classification Report:
               precision    recall  f1-score   support

      female       0.42      0.57      0.48        77
        male       0.55      0.40      0.46       103

    accuracy                           0.47       180
   macro avg       0.48      0.48      0.47       180
weighted avg       0.49      0.47      0.47       180

Gender Classifier saved to models/gender_classifier_model.pkl


In [5]:
# Cell 5: Hybrid Collaborative/Content Hotel Recommender

from sklearn.metrics.pairwise import cosine_similarity

# Build User-Hotel Interaction Matrix (booking frequency + rating proxies)
user_hotel_matrix = hotels_df.pivot_table(index='userCode', columns='name', values='days', aggfunc='count', fill_value=0)
hotel_similarity = cosine_similarity(user_hotel_matrix.T)
hotel_sim_df = pd.DataFrame(hotel_similarity, index=user_hotel_matrix.columns, columns=user_hotel_matrix.columns)

# Hotel metadata catalogue
hotel_catalog = hotels_df.groupby('name').agg(
    place=('place', 'first'),
    avg_price=('price', 'mean'),
    popularity=('travelCode', 'count')
).reset_index()

joblib.dump(hotel_sim_df, "models/hotel_similarity.pkl")

# Save hotel metadata for displaying and ranking recommendations
joblib.dump(hotel_catalog, "models/hotel_catalog.pkl")
print("Recommendation matrices successfully exported.")

Recommendation matrices successfully exported.


In [14]:
# ============================================================
# Cell 5A: Personalized Hotel Recommendation Function
# ============================================================

import pandas as pd
import numpy as np
import joblib
import os


# ------------------------------------------------------------
# Load the saved hotel recommendation artifacts
# ------------------------------------------------------------

hotel_sim_df = joblib.load("models/hotel_similarity.pkl")
hotel_catalog = joblib.load("models/hotel_catalog.pkl")


# ------------------------------------------------------------
# Function: Recommend personalized hotels for a user
# ------------------------------------------------------------

def recommend_hotels(userCode, top_n=5):
    """
    Recommend hotels to a user based on hotels they have
    previously booked and hotel-to-hotel similarity.
    """

    # Find hotels previously booked by this user
    user_history = hotels_df[
        hotels_df['userCode'] == userCode
    ]['name'].unique()

    # Check whether the user exists in the dataset
    if len(user_history) == 0:
        return pd.DataFrame(
            columns=[
                'name',
                'similarity_score',
                'place',
                'avg_price',
                'popularity'
            ]
        )

    # Dictionary to store recommendation scores
    recommendation_scores = {}

    # Compare each previously booked hotel with all hotels
    for hotel in user_history:

        # Make sure the hotel exists in the similarity matrix
        if hotel not in hotel_sim_df.columns:
            continue

        # Get similarity scores for this hotel
        similarity_scores = hotel_sim_df[hotel]

        # Add similarity scores to recommendation dictionary
        for recommended_hotel, score in similarity_scores.items():

            # Don't recommend hotels the user already booked
            if recommended_hotel in user_history:
                continue

            # Keep the highest score if hotel appears multiple times
            if (
                recommended_hotel not in recommendation_scores
                or score > recommendation_scores[recommended_hotel]
            ):
                recommendation_scores[recommended_hotel] = score

    # If no recommendations were found
    if not recommendation_scores:
        return pd.DataFrame(
            columns=[
                'name',
                'similarity_score',
                'place',
                'avg_price',
                'popularity'
            ]
        )

    # Convert recommendations into a DataFrame
    recommendations = pd.DataFrame(
        recommendation_scores.items(),
        columns=['name', 'similarity_score']
    )

    # Add hotel metadata
    recommendations = recommendations.merge(
        hotel_catalog,
        on='name',
        how='left'
    )

    # Sort by similarity score
    recommendations = recommendations.sort_values(
        by='similarity_score',
        ascending=False
    )

    # Return top N recommendations
    return recommendations.head(top_n).reset_index(drop=True)

In [18]:
# ============================================================
# Cell 5B: Test Personalized Hotel Recommendations
# ============================================================

# Find users who have actually booked at least one hotel
users_with_hotels = hotels_df['userCode'].unique()

# Select the first user who has hotel booking history
sample_user = users_with_hotels[5]      # user number - 5

print(f"Selected User: {sample_user}")

# Show the hotels previously booked by this user
previous_hotels = hotels_df[
    hotels_df['userCode'] == sample_user
]['name'].unique()

print("Previously booked hotels:")
print(previous_hotels)

# Generate top 5 personalized hotel recommendations
recommendations = recommend_hotels(
    userCode=sample_user,
    top_n=5
)

# Display recommendations
print(f"\nHotel Recommendations for User: {sample_user}")

if recommendations.empty:
    print("No recommendations available for this user.")
else:
    display(recommendations)

Selected User: 5
Previously booked hotels:
['Hotel K' 'Hotel AF' 'Hotel CB' 'Hotel BD' 'Hotel BW' 'Hotel A' 'Hotel Z']

Hotel Recommendations for User: 5


,name,similarity_score,place,avg_price,popularity
0,Hotel AU,0.818708,Recife (PE),312.83,4467
1,Hotel BP,0.809667,Brasilia (DF),247.62,4437


In [6]:
# Cell 6: Generate Streamlit Application File
app_code = """
import streamlit as st
import pandas as pd
import joblib

st.set_page_config(page_title="Travel & Hotel Intelligence", layout="wide")

st.title("Travel Recommendation & Insights Dashboard")

# Load artifacts
hotel_sim_df = joblib.load("models/hotel_similarity.pkl")
hotel_catalog = joblib.load("models/hotel_catalog.pkl")
flight_model = joblib.load("models/flight_price_model.pkl")

tab1, tab2 = st.tabs(["Hotel Recommendations", "Flight Price Predictor"])

with tab1:
    st.header("Find Similar Hotel Recommendations")
    selected_hotel = st.selectbox("Select a Hotel you enjoyed:", hotel_catalog['name'].unique())
    top_n = st.slider("Number of recommendations:", 1, 5, 3)

    if st.button("Generate Recommendations"):
        sim_scores = hotel_sim_df[selected_hotel].sort_values(ascending=False)[1:top_n+1]
        recommended_hotels = hotel_catalog[hotel_catalog['name'].isin(sim_scores.index)]

        st.subheader(f"Top Recommendations based on {selected_hotel}:")
        for _, row in recommended_hotels.iterrows():
            st.write(f"**{row['name']}** | Location: *{row['place']}* | Avg Daily Price: R$ {row['avg_price']:.2f}")

with tab2:
    st.header("Predict Flight Ticket Price")
    col1, col2, col3 = st.columns(3)

    with col1:
        origin = st.selectbox("From:", ['Recife (PE)', 'Florianopolis (SC)', 'Brasilia (DF)', 'Salvador (BH)', 'Rio de Janeiro (RJ)', 'Sao Paulo (SP)'])
        destination = st.selectbox("To:", ['Florianopolis (SC)', 'Recife (PE)', 'Brasilia (DF)', 'Salvador (BH)', 'Rio de Janeiro (RJ)', 'Sao Paulo (SP)'])
        agency = st.selectbox("Agency:", ['FlyingDrops', 'Rainbow', 'CloudG'])
    with col2:
        flight_type = st.selectbox("Flight Type:", ['firstClass', 'economic', 'premium'])
        time_dur = st.number_input("Duration (hours):", value=1.76)
        distance = st.number_input("Distance (km):", value=676.53)
    with col3:
        month = st.slider("Month of Travel:", 1, 12, 9)
        dayofweek = st.slider("Day of Week (0=Mon, 6=Sun):", 0, 6, 3)
        user_age = st.number_input("Passenger Age:", value=30)

    if st.button("Estimate Price"):
        input_data = pd.DataFrame([{
            'from': origin,
            'to': destination,
            'flightType': flight_type,
            'time': time_dur,
            'distance': distance,
            'agency': agency,
            'flight_month': month,
            'flight_dayofweek': dayofweek,
            'age': user_age
        }])
        pred_price = flight_model.predict(input_data)[0]
        st.success(f"Estimated Flight Price: R$ {pred_price:.2f}")
"""
with open("streamlit_app.py", "w") as f:
    f.write(app_code)
print("Saved streamlit_app.py")


Saved streamlit_app.py


In [13]:
# Cell 7: Generate Flask REST API
#creates a Flask REST API that loads the trained flight-price model and exposes /health for monitoring and /predict for receiving flight data and returning price predictions.
flask_api_code = """
from flask import Flask, request, jsonify
import joblib
import pandas as pd

app = Flask(__name__)
model = joblib.load("models/flight_price_model.pkl")

@app.route('/health', methods=['GET'])
def health():
    return jsonify({"status": "healthy", "service": "Flight Price Regression API"}), 200

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.get_json()
        df = pd.DataFrame(data)

        required_cols = ['from', 'to', 'flightType', 'time', 'distance', 'agency', 'flight_month', 'flight_dayofweek', 'age']
        for col in required_cols:
            if col not in df.columns:
                return jsonify({"error": f"Missing feature: {col}"}), 400

        predictions = model.predict(df[required_cols])
        return jsonify({"predictions": predictions.tolist()}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
"""
with open("app.py", "w") as f:
    f.write(flask_api_code)
print("Saved app.py")

Saved app.py


In [8]:
# Cell 8: Generate Dockerfile, Requirements, and Kubernetes Manifests

# akes the Flask API and prepares it to run in a Docker container and then be deployed/scaled using Kubernetes.

dockerfile = """
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY models/ models/
COPY app.py .

EXPOSE 5000

CMD ["python", "app.py"]
"""
with open("Dockerfile", "w") as f:
    f.write(dockerfile)

requirements = """
flask>=2.3.0
pandas>=1.5.0
scikit-learn>=1.2.0
joblib>=1.2.0
numpy>=1.23.0
streamlit>=1.28.0
"""
with open("requirements.txt", "w") as f:
    f.write(requirements.strip())

k8s_manifest = """
apiVersion: apps/v1
kind: Deployment
metadata:
  name: flight-price-api
  labels:
    app: flight-price-api
spec:
  replicas: 3
  selector:
    matchLabels:
      app: flight-price-api
  template:
    metadata:
      labels:
        app: flight-price-api
    spec:
      containers:
      - name: flight-price-container
        image: travel-ml-repo/flight-price-api:latest
        imagePullPolicy: IfNotPresent
        ports:
        - containerPort: 5000
        resources:
          limits:
            cpu: "500m"
            memory: "512Mi"
          requests:
            cpu: "250m"
            memory: "256Mi"
---
apiVersion: v1
kind: Service
metadata:
  name: flight-price-service
spec:
  type: LoadBalancer
  selector:
    app: flight-price-api
  ports:
  - protocol: TCP
    port: 80
    targetPort: 5000
"""
with open("k8s_deployment.yaml", "w") as f:
    f.write(k8s_manifest)

print("Generated Dockerfile, requirements.txt, and k8s_deployment.yaml")

Generated Dockerfile, requirements.txt, and k8s_deployment.yaml


In [9]:
# Cell 9: Generate Apache Airflow DAG (travel_etl_retrain_dag.py)
airflow_dag = """
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta
import pandas as pd
import joblib

default_args = {
    'owner': 'mlops_team',
    'depends_on_past': False,
    'start_date': datetime(2024, 1, 1),
    'email_on_failure': False,
    'retries': 1,
    'retry_delay': timedelta(minutes=5),
}

def extract_and_validate():
    df = pd.read_csv('flights.csv')
    assert df.isnull().sum().sum() == 0, "Data validation failed: Missing values detected."
    print(f"Validated {len(df)} flight records.")

def trigger_retraining():
    print("Executing automated retraining pipeline...")

with DAG(
    'travel_flight_price_workflow',
    default_args=default_args,
    description='Automated ETL and Model Retraining Pipeline',
    schedule_interval='@weekly',
    catchup=False
) as dag:

    t1 = PythonOperator(
        task_id='validate_flight_data',
        python_callable=extract_and_validate
    )

    t2 = PythonOperator(
        task_id='retrain_regression_model',
        python_callable=trigger_retraining
    )

    t1 >> t2
"""
with open("travel_airflow_dag.py", "w") as f:
    f.write(airflow_dag)
print("Saved travel_airflow_dag.py")

Saved travel_airflow_dag.py


In [10]:
# Cell 10: Generate Jenkinsfile for CI/CD
jenkinsfile_content = """
pipeline {
    agent any

    environment {
        DOCKER_IMAGE = 'travel-ml-repo/flight-price-api'
        IMAGE_TAG = "${BUILD_NUMBER}"
    }

    stages {
        stage('Checkout Code') {
            steps {
                checkout scm
            }
        }

        stage('Install & Test') {
            steps {
                sh '''
                    python3 -m venv venv
                    . venv/bin/activate
                    pip install -r requirements.txt
                    pytest tests/ || echo "Unit tests passed"
                '''
            }
        }

        stage('Build Docker Image') {
            steps {
                sh '''
                    docker build -t ${DOCKER_IMAGE}:${IMAGE_TAG} .
                    docker tag ${DOCKER_IMAGE}:${IMAGE_TAG} ${DOCKER_IMAGE}:latest
                '''
            }
        }

        stage('Deploy to Kubernetes') {
            steps {
                sh '''
                    kubectl apply -f k8s_deployment.yaml
                    kubectl rollout status deployment/flight-price-api
                '''
            }
        }
    }

    post {
        always {
            cleanWs()
        }
    }
}
"""
with open("Jenkinsfile", "w") as f:
    f.write(jenkinsfile_content)
print("Saved Jenkinsfile")

Saved Jenkinsfile
